In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D
from PIL import ImageFile
import pandas as pd
import numpy as np
import os

In [2]:
from tensorflow.keras import layers, models

In [3]:
base_model = ResNet50(weights="imagenet", include_top=False)
base_model.trainable = False

model = models.Sequential()
model.add(base_model)
model.add(layers.GlobalAveragePooling2D())
model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, None, None,     │    23,587,712 │
│                                 │ 2048)                  │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,587,712 (89.98 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 23,587,712 (89.98 MB)

In [4]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [5]:
ImageFile.LOAD_TRUNCATED_IMAGES = True
trainPath='/content/gdrive/MyDrive/train'
valiPath='/content/gdrive/MyDrive/val'

In [6]:
trainImage = ImageDataGenerator(rescale=1./255).flow_from_directory(trainPath,target_size=(224,224),batch_size=32,class_mode='categorical')
valiImage = ImageDataGenerator(rescale=1./255).flow_from_directory(valiPath,target_size=(224,224),batch_size=32,class_mode='categorical')

Found 2872 images belonging to 18 classes.
Found 761 images belonging to 18 classes.


In [7]:
trainLabel = []
valiLabel = []

for folder in sorted(os.listdir(trainPath)):
    folder_path = os.path.join(trainPath, folder)
    if os.path.isdir(folder_path):
        label = folder
        for file in sorted(os.listdir(folder_path)):
            if os.path.isfile(os.path.join(folder_path, file)):
                trainLabel.append((label, label+"/"+file))

for folder in sorted(os.listdir(valiPath)):
    folder_path = os.path.join(valiPath, folder)
    if os.path.isdir(folder_path):
        label = folder
        for file in sorted(os.listdir(folder_path)):
            if os.path.isfile(os.path.join(folder_path, file)):
                valiLabel.append((label, label+"/"+file))


In [8]:
train = pd.DataFrame(trainLabel, columns=['label', 'pathname'])
vali = pd.DataFrame(valiLabel, columns=['label', 'pathname'])

train.to_csv('trainLabel.csv', index=False)
vali.to_csv('valiLabel.csv', index=False)

In [9]:
train_features = model.predict(trainImage, verbose=1)
val_features = model.predict(valiImage, verbose=1)

train_filenames = trainImage.filenames
val_filenames = valiImage.filenames

df_train = pd.DataFrame(train_features)
df_val = pd.DataFrame(val_features)

df_train.insert(0, "filename", train_filenames)
df_val.insert(0, "filename", val_filenames)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


90/90 ━━━━━━━━━━━━━━━━━━━━ 1834s 21s/step
24/24 ━━━━━━━━━━━━━━━━━━━━ 467s 20s/step


In [10]:
df_train.to_csv("trainFeature.csv", index=False)
df_val.to_csv("valiFeature.csv", index=False)

In [11]:
import joblib

filename = 'restnet50.sav'
joblib.dump(model, filename)

['restnet50.sav']

# End of restnet50

In [97]:
trainFeature= pd.read_csv('trainFeature.csv')
valiFeature=pd.read_csv('valiFeature.csv')

trainFilename=trainFeature['filename']
valiFilename=valiFeature['filename']

trainFeature=trainFeature.drop(['filename'],axis=1)
valiFeature=valiFeature.drop(['filename'],axis=1)

In [98]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
scaler.fit(trainFeature)

trainFeature = scaler.transform(trainFeature)
valiFeature = scaler.transform(valiFeature)

In [99]:
from sklearn.decomposition import PCA

pca = PCA(n_components=512)
trainFeature= pca.fit_transform(trainFeature)
valiFeature= pca.transform(valiFeature)

In [51]:
trainLabel= pd.read_csv('trainLabel.csv')
valiLabel = pd.read_csv('valiLabel.csv')

trainLabel=trainLabel.drop(['pathname'],axis=1)
valiLabel=valiLabel.drop(['pathname'],axis=1)

In [52]:
trainLabel=pd.get_dummies(trainLabel['label'])
valiLabel=pd.get_dummies(valiLabel['label'])
valiLabel=valiLabel.reindex(columns=trainLabel.columns, fill_value=0)

In [53]:
from tensorflow.keras.optimizers import Adam

In [82]:
model = models.Sequential()
model.add(layers.Dense(2048, input_shape=(512,)))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))

model.add(layers.Dense(4096))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))
model.add(layers.Dropout(0.3))

model.add(layers.Dense(2048))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))
model.add(layers.Dropout(0.3))

model.add(layers.Dense(512))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))
model.add(layers.Dropout(0.3))

model.add(layers.Dense(256))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))

model.add(layers.Dense(128))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))

model.add(layers.Dense(64))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))

model.add(layers.Dense(32))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))

model.add(layers.Dense(18, activation='softmax'))
model.summary()

adamm = Adam(learning_rate=0.00025)
model.compile(optimizer=adamm, loss='categorical_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_80 (Dense)                │ (None, 2048)           │     1,050,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_65          │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_45 (Activation)      │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_81 (Dense)                │ (None, 4096)           │     8,392,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_66          │ (None, 4096)           │        16,384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_46 (Activation)      │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_53 (Dropout)            │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_82 (Dense)                │ (None, 2048)           │     8,390,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_67          │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_47 (Activation)      │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_54 (Dropout)            │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_83 (Dense)                │ (None, 512)            │     1,049,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_68          │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_48 (Activation)      │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_55 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_84 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_69          │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_49 (Activation)      │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_85 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_70          │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_50 (Activation)      │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_86 (Dense)                │ (None, 64)             │         8,25

 Total params: 19,094,962 (72.84 MB)

 Trainable params: 19,076,594 (72.77 MB)

 Non-trainable params: 18,368 (71.75 KB)

In [83]:
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=20,
    restore_best_weights=True
)

In [84]:
history = model.fit(trainFeature, trainLabel, epochs=300,
                    validation_data=(valiFeature, valiLabel),
                    callbacks=[early_stop])

Epoch 1/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 17s 71ms/step - accuracy: 0.0662 - loss: 3.1409 - val_accuracy: 0.0460 - val_loss: 2.9054
Epoch 2/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.0799 - loss: 2.9874 - val_accuracy: 0.0539 - val_loss: 2.9502
Epoch 3/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.0858 - loss: 2.9159 - val_accuracy: 0.0631 - val_loss: 2.9721
Epoch 4/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.0925 - loss: 2.8828 - val_accuracy: 0.0657 - val_loss: 2.9829
Epoch 5/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.1124 - loss: 2.8464 - val_accuracy: 0.0618 - val_loss: 2.9985
Epoch 6/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.1165 - loss: 2.8192 - val_accuracy: 0.0526 - val_loss: 3.0090
Epoch 7/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.1208 - loss: 2.7912 - val_accuracy: 0.0670 - val_loss: 3.0233
Epoch 8/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.1187 - loss: 2.7788 - val_accuracy: 0.055

In [87]:
new_model = models.Sequential(model.layers[:-1])

for old_layer, new_layer in zip(model.layers[:-1], new_model.layers):
    new_layer.set_weights(old_layer.get_weights())

In [112]:
new_model.summary()

Model: "sequential_17"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_80 (Dense)                │ (None, 2048)           │     1,050,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_65          │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_45 (Activation)      │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_81 (Dense)                │ (None, 4096)           │     8,392,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_66          │ (None, 4096)           │        16,384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_46 (Activation)      │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_53 (Dropout)            │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_82 (Dense)                │ (None, 2048)           │     8,390,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_67          │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_47 (Activation)      │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_54 (Dropout)            │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_83 (Dense)                │ (None, 512)            │     1,049,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_68          │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_48 (Activation)      │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_55 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_84 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_69          │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_49 (Activation)      │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_85 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_70          │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_50 (Activation)      │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_86 (Dense)                │ (None, 64)             │         8,25

 Total params: 19,094,368 (72.84 MB)

 Trainable params: 19,076,000 (72.77 MB)

 Non-trainable params: 18,368 (71.75 KB)

In [88]:
import joblib

filename = 'CNN.sav'
joblib.dump(new_model, filename)

['CNN.sav']

In [102]:
joblib.dump(scaler, 'minmaxScaler.pkl')
joblib.dump(pca, 'pca.pkl')

['pca.pkl']

In [103]:
trainFeature= pd.read_csv('trainFeature.csv')
valiFeature=pd.read_csv('valiFeature.csv')

trainFilename=trainFeature['filename']
valiFilename=valiFeature['filename']

trainFeature=trainFeature.drop(['filename'],axis=1)
valiFeature=valiFeature.drop(['filename'],axis=1)

In [104]:
trainFeature = scaler.transform(trainFeature)
valiFeature = scaler.transform(valiFeature)

In [106]:
trainFeature= pca.transform(trainFeature)
valiFeature= pca.transform(valiFeature)

In [107]:
trainFeature = model.predict(trainFeature, verbose=1)
valiFeature = model.predict(valiFeature, verbose=1)

90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step


In [108]:
df_train = pd.DataFrame(trainFeature)
df_val = pd.DataFrame(valiFeature)

df_train.insert(0, "filename", trainFilename)
df_val.insert(0, "filename", valiFilename)

In [110]:
df_train.to_csv("trainVectors.csv", index=False)
df_val.to_csv("valVectors.csv", index=False)